# Tangram tutorial with squidpy

The `mode="cells"` part of [Tangram's tutorial](https://github.com/broadinstitute/Tangram/blob/4c68995a418f41dc8caef567598c4d9b47781a13/tutorial_tangram_with_squidpy.ipynb), which maps the
`sc_mouse_cortex` scRNA-seq reference onto the cortex of the `visium_fluo` Visium section, written against
[`squidpy.experimental.tl.impute_tangram`](https://github.com/scverse/squidpy/blob/main/src/squidpy/experimental/tl/_impute_tangram.py).

The port maps cells and projects genes, so the tutorial's gene plots and its score of every gene are reproduced
here. Its cell-type projection, `mode="constrained"`, segmentation and deconvolution are not part of the port.
The last section runs upstream Tangram from this repository's pinned copy on the same inputs.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

import squidpy as sq

In [ ]:
adata_st = sq.datasets.visium_fluo_adata_crop()
adata_st = adata_st[adata_st.obs.cluster.isin([f"Cortex_{i}" for i in np.arange(1, 5)])].copy()
adata_sc = sq.datasets.sc_mouse_cortex()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(20, 5))
sc.pl.spatial(adata_st, color="cluster", alpha=0.7, frameon=False, show=False, ax=axs[0])
sc.pl.umap(adata_sc, color="cell_subclass", size=10, frameon=False, show=False, ax=axs[1])
plt.tight_layout()

The tutorial trains on the top 100 marker genes of each cell subclass.

In [ ]:
sc.tl.rank_genes_groups(adata_sc, groupby="cell_subclass", use_raw=False)
markers_df = pd.DataFrame(adata_sc.uns["rank_genes_groups"]["names"]).iloc[0:100, :]
markers = list(np.unique(markers_df.melt().value.values))
len(markers)

`tg.pp_adatas` lowercases every gene name, so that names match across species capitalizations; `impute_tangram`
compares names as given, so they are lowercased here instead.

`impute_tangram` trains on every gene the two datasets share, so the query passed to it is the section restricted
to the marker genes. One consequence: its `"rna_count_based"` prior counts those genes only, where `tg.pp_adatas`
counts every gene of the section.

In [ ]:
for adata in (adata_sc, adata_st):
    adata.var_names = adata.var_names.str.lower()
    adata.var_names_make_unique()

training = sorted(set(map(str.lower, markers)) & set(adata_st.var_names))
shared = sorted(set(adata_sc.var_names) & set(adata_st.var_names))
f"{len(training)} training genes, {len(shared) - len(training)} test genes"

In [ ]:
adata_st.obsm["tangram"] = sq.experimental.tl.impute_tangram(
    adata_sc,
    adata_st[:, training].copy(),
    genes=shared,
    density_prior="rna_count_based",
    num_epochs=500,
    rng=0,
    inplace=False,
)

## Measured and predicted genes

`tg.plot_genes_sc` shows each gene measured and predicted on the tissue, each scaled to its own range. First three
training genes, then the test genes the tutorial picks.

In [ ]:
def plot_genes(genes):
    measured = adata_st[:, genes].to_df().add_suffix(" (measured)")
    predicted = adata_st.obsm["tangram"][genes].add_suffix(" (predicted)")
    adata_st.obs[measured.columns] = measured
    adata_st.obs[predicted.columns] = predicted
    columns = [c for pair in zip(measured.columns, predicted.columns, strict=True) for c in pair]
    sc.pl.spatial(adata_st, color=columns, ncols=2, frameon=False, alpha_img=0.5, cmap="inferno")
    adata_st.obs.drop(columns=columns, inplace=True)


plot_genes(["rragb", "trim17", "eno1b"])

In [ ]:
plot_genes(['loc102633833', 'gm5700', 'gm8292'])

In [ ]:
plot_genes(['tfap2b', 'zic4', 'cd34', 'rasal1', 'gm33027', 'gm5431'])

## Scores of all genes

`tg.compare_spatial_geneexp` scores each gene by the cosine similarity of its measured and predicted expression
across spots, and plots it against how sparse the gene is in the section.

In [ ]:
def scores(predicted):
    measured = adata_st[:, predicted.columns].to_df()
    norms = np.linalg.norm(measured, axis=0) * np.linalg.norm(predicted, axis=0)
    with np.errstate(invalid="ignore", divide="ignore"):
        return pd.Series((measured.to_numpy() * predicted.to_numpy()).sum(0) / norms, index=predicted.columns)


df_all_genes = pd.DataFrame(
    {
        "score": scores(adata_st.obsm["tangram"]),
        "sparsity_sp": 1 - np.asarray((adata_st[:, shared].X > 0).mean(0)).ravel(),
        "is_training": [g in training for g in shared],
    }
)
fig, ax = plt.subplots(figsize=(5, 5))
for is_training, group in df_all_genes.groupby("is_training"):
    ax.scatter(group.score, group.sparsity_sp, s=5, alpha=0.5, label="training" if is_training else "test")
ax.set(xlabel="score", ylabel="spatial sparsity")
ax.legend()
df_all_genes.groupby("is_training").score.describe()

## Against upstream

Upstream Tangram from this repository's pinned copy (`vendor/Tangram`), on the same reference, the same
marker-restricted section and the same settings, through `squidpy_ports.tangram.upstream.run`: the tutorial's
`pp_adatas` → `map_cells_to_space` → `project_genes`. Each implementation draws its own random start, so the two
mappings differ in their details; the gene scores they reach are what should agree.

In [ ]:
from squidpy_ports._common import Imputation
from squidpy_ports.tangram import upstream

test = [g for g in shared if g not in set(training)]
data = Imputation(
    ref=adata_sc[:, training + test].X.toarray(),
    query=adata_st[:, training].X.toarray(),
    truth=adata_st[:, test].X.toarray(),
    genes=training + test,
)
upstream_res = upstream.run(upstream.load(), data, num_epochs=500, density_prior="rna_count_based", random_state=0)
upstream_scores = scores(pd.DataFrame(upstream_res["imputed"], index=adata_st.obs_names, columns=test))

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(upstream_scores, df_all_genes.score[test], s=5, alpha=0.5)
ax.axline((0, 0), slope=1, color="grey", lw=1)
ax.set(xlabel="upstream score", ylabel="squidpy score", title="test genes")
print(f"median test score: squidpy {df_all_genes.score[test].median():.3f}, upstream {upstream_scores.median():.3f}")
print(f"correlation of the per-gene scores: {np.corrcoef(upstream_scores.fillna(0), df_all_genes.score[test].fillna(0))[0, 1]:.3f}")